# 🚁 RT-DETR Drone Detection (Kaggle Pipeline)

**Model:** RT-DETR-L (Transformer)
**Hardware:** Kaggle GPU (P100/T4)
**Dataset:** Input dataset from Kaggle

---

### 1. Setup Environment

In [ ]:
!pip install -q ultralytics

import os
import yaml
import glob

print("✅ Environment ready!")

### 2. Configure Dataset Paths

**Note:** On Kaggle, your input dataset is automatically available at `/kaggle/input/`

In [ ]:
# Kaggle automatically mounts datasets at /kaggle/input/
# Update this to match your dataset name on Kaggle
DATASET_NAME = 'drone-detection'  # Change this to your actual dataset name

# Find the dataset root
dataset_base = f'/kaggle/input/{DATASET_NAME}'

if not os.path.exists(dataset_base):
    print(f"⚠️ Dataset not found at {dataset_base}")
    print("Available datasets:")
    !ls /kaggle/input/
    raise FileNotFoundError(f"Please update DATASET_NAME variable to match your Kaggle dataset")

print(f"📍 Dataset found at: {dataset_base}")

# Find the data.yaml file
yaml_files = glob.glob(f'{dataset_base}/**/data.yaml', recursive=True)
if not yaml_files:
    yaml_files = glob.glob(f'{dataset_base}/**/*.yaml', recursive=True)

if not yaml_files:
    raise FileNotFoundError("Could not find data.yaml in the dataset!")

yaml_path = yaml_files[0]
dataset_root = os.path.dirname(yaml_path)
print(f"📍 Found config at: {yaml_path}")

# Read and display current config
with open(yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)
    
print("\n📋 Current dataset config:")
print(f"  Classes: {data_config.get('nc', 'Unknown')}")
print(f"  Names: {data_config.get('names', 'Unknown')}")
print(f"  Train: {data_config.get('train', 'Unknown')}")
print(f"  Val: {data_config.get('val', 'Unknown')}")

### 3. Create Working YAML Config

Since Kaggle datasets are read-only, we'll create a working copy

In [ ]:
# Create a working directory for outputs
!mkdir -p /kaggle/working/config

# Create a new YAML with absolute paths for Kaggle
working_yaml = '/kaggle/working/config/data.yaml'

# Update paths to be absolute
data_config['path'] = dataset_root
data_config['train'] = 'train/images'
data_config['val'] = 'valid/images'

# Add test if it exists
if os.path.exists(os.path.join(dataset_root, 'test/images')):
    data_config['test'] = 'test/images'

# Write the working config
with open(working_yaml, 'w') as f:
    yaml.dump(data_config, f)

print(f"✅ Created working config at: {working_yaml}")
print("\n📋 Final config:")
with open(working_yaml, 'r') as f:
    print(f.read())

### 4. Train RT-DETR Model

**Training Parameters:**
- Model: RT-DETR-L (Large)
- Epochs: 30
- Image Size: 640
- Batch Size: 8 (optimized for Kaggle GPU)

In [ ]:
from ultralytics import RTDETR

# Load RT-DETR-L (Large) Model
model = RTDETR('rtdetr-l.pt') 

print("🔥 STARTING TRAINING (30 Epochs)...")
print(f"📊 Using config: {working_yaml}\n")

# Train
results = model.train(
    data=working_yaml,
    epochs=30,
    imgsz=640,
    batch=8,                    # Optimized for Kaggle GPU
    device=0,                   # Use GPU
    workers=2,
    project='/kaggle/working/runs/train',
    name='rtdetr_drone',
    exist_ok=True,
    amp=True,                   # Mixed Precision enabled
    patience=10,                # Early stopping
    save=True,
    save_period=5,              # Save checkpoint every 5 epochs
    cache=False,                # Don't cache (saves memory)
    verbose=True
)

print("\n✅ Training complete!")

### 5. Validate Model

In [ ]:
print("📊 Running validation...\n")

# Validate on validation set
metrics = model.val()

print("\n📈 Validation Results:")
print(f"  mAP@0.5: {metrics.box.map50:.3f}")
print(f"  mAP@0.5:0.95: {metrics.box.map:.3f}")
print(f"  Precision: {metrics.box.mp:.3f}")
print(f"  Recall: {metrics.box.mr:.3f}")

### 6. Test on Random Images

In [ ]:
from IPython.display import Image, display
import random

print("📊 Generating Predictions on Test Images...\n")

# Get test images
test_dir = os.path.join(dataset_root, 'valid/images')  # or 'test/images' if available
test_images = glob.glob(f"{test_dir}/*.jpg") + glob.glob(f"{test_dir}/*.png")

if test_images:
    # Pick 6 random images
    sample_images = random.sample(test_images, min(6, len(test_images)))
    
    print(f"Testing on {len(sample_images)} random images...\n")
    
    # Run predictions
    results_pred = model.predict(
        sample_images, 
        save=True, 
        conf=0.5,
        project='/kaggle/working/predictions',
        name='test_results'
    )
    
    print("\n✅ Predictions saved to: /kaggle/working/predictions/test_results/")
    
    # Display first result
    if results_pred:
        pred_path = '/kaggle/working/predictions/test_results/' + os.path.basename(sample_images[0])
        if os.path.exists(pred_path):
            print("\n👁️ Sample Prediction:")
            display(Image(filename=pred_path, width=600))
else:
    print("⚠️ No test images found.")

### 7. Display Training Results

In [ ]:
# Show training metrics
results_png = '/kaggle/working/runs/train/rtdetr_drone/results.png'
if os.path.exists(results_png):
    print("📈 Training Metrics:")
    display(Image(filename=results_png, width=1000))
else:
    print("⚠️ Results plot not found.")

# Show confusion matrix if available
confusion_matrix = '/kaggle/working/runs/train/rtdetr_drone/confusion_matrix.png'
if os.path.exists(confusion_matrix):
    print("\n🎯 Confusion Matrix:")
    display(Image(filename=confusion_matrix, width=600))

### 8. Save Model Weights

The best model weights are automatically saved in `/kaggle/working/runs/train/rtdetr_drone/weights/`

In [ ]:
print("📦 Model weights saved at:")
print("  Best: /kaggle/working/runs/train/rtdetr_drone/weights/best.pt")
print("  Last: /kaggle/working/runs/train/rtdetr_drone/weights/last.pt")

# List all files in the weights directory
weights_dir = '/kaggle/working/runs/train/rtdetr_drone/weights/'
if os.path.exists(weights_dir):
    print("\n📁 Available weight files:")
    !ls -lh {weights_dir}
else:
    print("⚠️ Weights directory not found.")